# Lesson 20: Hugging Face — the app store for AI models 🤗
> 🎓 **Advanced track** — best for Years 11–13. Students should be comfortable with Python functions, lists and dictionaries.

**Learning goals**
1. Find models on the **Hugging Face Hub** and read a **model card**
2. Use **pipelines** for a dozen tasks in one line each
3. Go one level deeper with `AutoTokenizer` and `AutoModel`
4. Compare models, and use the Hub's datasets

**⏱️ Timing:** the Hub 10' | pipelines 20' | Auto classes 15' | compare models 15'

> ⚙️ **Before you start:** go to **Runtime → Change runtime type → T4 GPU**, then run the cells from top to bottom.

## 🏪 An analogy: an app store for AI
Instead of training everything from scratch (which can cost millions), you download a **pretrained model** that someone else trained, just like installing an app.
- **Hugging Face Hub** (huggingface.co): hundreds of thousands of free models and datasets
- **`transformers` library**: the Python code to download and run them
- **Model card**: the model's "app description" — what it does, what data it learned from, its limits and biases, and its **licence** (what you're allowed to use it for)

You've already used it in Lessons 2, 5, 7, 8 and 14 (`pipeline(...)`, `AutoModel...`). Today we learn it properly.

## Part 1: Explore the Hub 🔎
Open https://huggingface.co/models in a new tab and try the filters on the left: **Tasks**, **Languages**, **Licenses**.

### 🧩 Exercise 1: Model-card detective
Open the card for `distilbert-base-uncased-finetuned-sst-2-english` and answer:
1. What task does it do? 2. What data was it trained on? 3. What licence does it have? 4. Does the card mention any **biases or limitations**?

You can also search the Hub from code:

In [ ]:
from huggingface_hub import HfApi
api = HfApi()
for m in api.list_models(pipeline_tag="text-classification", sort="downloads", limit=5):
    print(f"{m.id:60} downloads: {m.downloads}")

## Part 2: Pipelines — one line per task 🚀
A **pipeline** bundles three steps: **tokenizer** (text → numbers) → **model** → **post-processing** (numbers → answer).

In [ ]:
from transformers import pipeline
import torch
device = 0 if torch.cuda.is_available() else -1

sentiment = pipeline("sentiment-analysis", model="distilbert-base-uncased-finetuned-sst-2-english", device=device)
print(sentiment(["This lesson is awesome!", "I lost my homework again."]))

In [ ]:
zero_shot = pipeline("zero-shot-classification", model="facebook/bart-large-mnli", device=device)
print(zero_shot("The All Blacks won the match 32–10 on Saturday.",
                candidate_labels=["sport", "politics", "science", "cooking"]))

In [ ]:
ner = pipeline("ner", model="dslim/bert-base-NER", aggregation_strategy="simple", device=device)
for e in ner("Marie Curie worked in Paris and won two Nobel Prizes."):
    print(f"{e['word']:12} → {e['entity_group']}  ({e['score']:.0%})")

> 📢 **Real-world lesson:** in 2025–26 the `transformers` library (version 5) **removed** some older one-line tasks such as `"summarization"`, `"translation"` and `"question-answering"`. Why? Instruction-following LLMs now do those jobs well, so many people use one chat model instead of many small specialist models. You can still run the specialist models with the `Auto` classes (Part 3). AI libraries change fast — checking the docs is part of the job!

In [ ]:
# Instead of a "summarization" pipeline, use a small instruction-following LLM via "text-generation"
chat = pipeline("text-generation", model="Qwen/Qwen2.5-0.5B-Instruct", device=device)
article = """The school science fair took place on Friday. More than 60 students presented projects,
ranging from water filters made of sand and charcoal to robots that sort recycling. The winning project,
by two Year 10 students, used a cheap sensor to measure air quality in classrooms and showed that opening
windows for ten minutes cut carbon dioxide levels by half. The judges praised the clear experiment design."""
messages = [{"role": "user", "content": "Summarise in one sentence:\n" + article}]
print(chat(messages, max_new_tokens=60)[0]["generated_text"][-1]["content"])

In [ ]:
# Zero-shot IMAGE classification with CLIP: it matches pictures to any labels you type
from PIL import Image
from skimage import data
image_clf = pipeline("zero-shot-image-classification", model="openai/clip-vit-base-patch32", device=device)
img = Image.fromarray(data.chelsea())     # a photo that ships with scikit-image
display(img.resize((200, 150)))
for r in image_clf(img, candidate_labels=["a cat", "a dog", "a rocket", "a cup of coffee"]):
    print(f"{r['label']:16} {r['score']:.0%}")

### 🧩 Exercise 2: Pipeline playground
1. Give `zero_shot` a sentence and your own labels (e.g. the subjects you study).
2. Try `image_clf` on `data.astronaut()`, `data.coffee()` or `data.rocket()` with your own labels. Can you trick it?
3. Find **one more pipeline task** on the Hub (e.g. `fill-mask`, `text-to-speech`, `object-detection`) and get it working.

> 💡 If a pipeline says a task name doesn't exist, the library may have renamed it — check the model card's code example. Libraries change quickly; reading docs is a real AI-engineer skill!

## Part 3: Under the hood — `AutoTokenizer` + `AutoModel` 🔧
`Auto` classes read the model's config and pick the right architecture for you.

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

name = "distilbert-base-uncased-finetuned-sst-2-english"
tokenizer = AutoTokenizer.from_pretrained(name)
model = AutoModelForSequenceClassification.from_pretrained(name)

text = "The movie was surprisingly good!"
tokens = tokenizer(text, return_tensors="pt")
print("tokens:", tokenizer.convert_ids_to_tokens(tokens["input_ids"][0]))
print("ids:   ", tokens["input_ids"][0].tolist())

with torch.no_grad():
    logits = model(**tokens).logits
probs = torch.softmax(logits, dim=1)[0]
for label_id, p in enumerate(probs):
    print(model.config.id2label[label_id], f"{p:.1%}")
print("\nThis model has", f"{sum(p.numel() for p in model.parameters()):,}", "parameters")

`[CLS]` and `[SEP]` are **special tokens** BERT-style models add to mark the start and the end. The pipeline did all of this for you.

### Running a specialist model by hand: extractive question answering
This model doesn't write an answer — it **points** at the start and end of the answer inside the text you give it.

In [ ]:
from transformers import AutoModelForQuestionAnswering
qa_name = "distilbert-base-cased-distilled-squad"
qa_tok = AutoTokenizer.from_pretrained(qa_name)
qa_model = AutoModelForQuestionAnswering.from_pretrained(qa_name)

def answer(question, context):
    inputs = qa_tok(question, context, return_tensors="pt")
    with torch.no_grad():
        out = qa_model(**inputs)
    start, end = out.start_logits.argmax().item(), out.end_logits.argmax().item()
    return qa_tok.decode(inputs["input_ids"][0][start:end + 1])

context = "Kiwi are flightless birds native to New Zealand. They are nocturnal and lay very large eggs."
for q in ["When are kiwi active?", "Where do kiwi come from?", "What colour are kiwi?"]:
    print(q, "→", answer(q, context))

Notice the last question: the answer isn't in the text, but the model still points at *something*. Specialist models can be confidently wrong too!

## Part 4: Compare models for the same task ⚖️
Different models trained on different data give different answers. Picking the right model is part of the job.

In [ ]:
models = ["distilbert-base-uncased-finetuned-sst-2-english",
          "cardiffnlp/twitter-roberta-base-sentiment-latest"]
tests = ["lol this is sick 🔥", "Not bad at all.", "The food was cold and the waiter ignored us."]
for m in models:
    clf = pipeline("sentiment-analysis", model=m, device=device)
    print(f"\n== {m}")
    for t in tests:
        r = clf(t)[0]
        print(f"   {r['label']:10} {r['score']:.0%}  {t}")

### 🧩 Exercise 3
1. Which model understands **slang and emoji** better? Look at its model card: what data was it trained on? Does that explain it?
2. One model has 2 labels, the other has 3. Why might "neutral" be useful?

## Part 5: Datasets from the Hub 📦

In [ ]:
from datasets import load_dataset
ds = load_dataset("cornell-movie-review-data/rotten_tomatoes", split="test[:5]")
print(ds)
for row in ds:
    print(row["label"], "|", row["text"][:80])

### 🚀 Challenge
Measure the accuracy of both sentiment models on 200 reviews from `cornell-movie-review-data/rotten_tomatoes` (label 1 = positive, 0 = negative). Which model wins on movie reviews? Is that the same winner as for slang?

## 📌 Summary
- **Hub** = app store of pretrained models and datasets; always read the **model card** (data, limits, licence)
- **Pipeline** = tokenizer + model + post-processing in one line
- **AutoTokenizer / AutoModel** give you control over each step
- The "best" model depends on **your** data — test before you trust